In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer: funkcja Qiskit firmy Global Data Quantum


*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** Funkcje Qiskit to funkcjonalność eksperymentalna dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (przez IBM Quantum Platform API). Są w statusie podglądu i mogą ulec zmianie.
## Przegląd
Quantum Portfolio Optimizer to funkcja Qiskit, która rozwiązuje problem dynamicznej optymalizacji portfela – standardowy problem w finansach polegający na okresowym równoważeniu inwestycji w zestawie aktywów w celu maksymalizacji zwrotów i minimalizacji ryzyka. Dzięki zastosowaniu najnowocześniejszych technik optymalizacji kwantowej funkcja upraszcza ten proces, umożliwiając użytkownikom nieposiadającym wiedzy z zakresu informatyki kwantowej skorzystanie z jej zalet przy wyznaczaniu optymalnych ścieżek inwestycyjnych. To narzędzie jest idealne dla zarządzających portfelami, badaczy z obszaru finansów ilościowych oraz indywidualnych inwestorów – umożliwia wsteczne testowanie strategii inwestycyjnych w optymalizacji portfela.
## Opis funkcji
Funkcja Quantum Portfolio Optimizer wykorzystuje algorytm Variational Quantum Eigensolver (VQE) do rozwiązywania problemu kwadratowej niezwiązanej optymalizacji binarnej (QUBO), odpowiadającego problemowi dynamicznej optymalizacji portfela. Wystarczy, że podasz dane o cenach aktywów i zdefiniujesz ograniczenia inwestycyjne – funkcja uruchomi kwantowy proces optymalizacji i zwróci zestaw zoptymalizowanych ścieżek inwestycyjnych.

Proces składa się z czterech głównych etapów. Najpierw dane wejściowe są odwzorowywane na problem kompatybilny z kwantowym przetwarzaniem: budowany jest QUBO dla problemu dynamicznej optymalizacji portfela, a następnie przekształcany w operator kwantowy (hamiltonian Isinga). Następnie problem wejściowy oraz algorytm VQE są dostosowywane do uruchomienia na sprzęcie kwantowym. Algorytm VQE jest uruchamiany na sprzęcie kwantowym, po czym wyniki są poddawane przetwarzaniu końcowemu w celu wyznaczenia optymalnych ścieżek inwestycyjnych. System zawiera również przetwarzanie końcowe uwzględniające szumy (oparte na [SQD](/guides/qiskit-addons-sqd)), aby zmaksymalizować jakość wyników.

Ta funkcja Qiskit jest oparta na [opublikowanym artykule naukowym](https://arxiv.org/abs/2412.19150) firmy Global Data Quantum.
![Wizualizacja przepływu pracy funkcji](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Pierwsze kroki
Uwierzytelnij się przy użyciu swojego [klucza API](https://quantum.cloud.ibm.com/) i wybierz funkcję Qiskit w następujący sposób. (Ten fragment kodu zakłada, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w swoim lokalnym środowisku.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## Przykład: Dynamiczna optymalizacja portfela z siedmioma aktywami
Ten przykład pokazuje, jak uruchomić funkcję dynamicznej optymalizacji portfela (DPO) i dostosować jej ustawienia w celu uzyskania optymalnej wydajności. Zawiera szczegółowe kroki pozwalające na precyzyjne dostrojenie parametrów w celu osiągnięcia pożądanych wyników.

Ten przypadek obejmuje siedem aktywów, cztery kroki czasowe i cztery Qubity rozdzielczości, co w sumie wymaga 112 Qubitów.
### 1. Wczytaj aktywa wchodzące w skład portfela.
Jeśli wszystkie aktywa portfela są przechowywane w folderze pod określoną ścieżką, możesz załadować je do obiektu `pandas.DataFrame` i przekonwertować do formatu `dict` za pomocą poniższej funkcji.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them
    into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

W tym przykładzie użyto aktywów [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) oraz [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Poniższy rysunek ilustruje dane użyte w tym przykładzie, przedstawiając dzienną ewolucję cen zamknięcia aktywów od 1 stycznia do 1 września 2023 roku.

W tym przykładzie, aby zapewnić jednolitość dat, wypełniliśmy dni bez notowań ceną zamknięcia z poprzedniego dostępnego dnia. Zastosowaliśmy ten krok, ponieważ wybrane aktywa pochodzą z różnych rynków o zróżnicowanych dniach sesyjnych, co sprawia, że standaryzacja zbioru danych jest niezbędna dla zachowania spójności.
![Visualization of the historical data of the assets](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. Zdefiniuj problem.
Zdefiniuj specyfikację problemu, konfigurując parametry w słowniku `qubo_settings`.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. Zdefiniuj ustawienia optymalizatora i ansatzu (Opcjonalne)
Opcjonalnie zdefiniuj szczegółowe wymagania dotyczące procesu optymalizacji, w tym wybór optymalizatora i jego parametrów, a także specyfikację prymitywu i jego konfiguracji.

W przypadku Tailored Ansatz wybrany rozmiar populacji był oparty na wcześniejszych eksperymentach, które wykazały, że ta wartość zapewnia stabilną i efektywną optymalizację.

W przypadku Real Amplitudes Ansatz możesz kierować się liniową zależnością pomiędzy ``population_size`` a liczbą Qubitów w Circuit. Jako przybliżona zasada praktyczna zaleca się stosowanie **minimalnego** ``population_size ~ 0.8 * n_qubits`` dla ansatzu `real_amplitudes`.

Oczekuje się, że Optimized Real Amplitudes będzie miał lepszą wydajność optymalizacji niż ansatz Real Amplitudes. Jednak liczba zmiennych do zoptymalizowania w tym ansatzu rośnie znacznie szybciej niż w przypadku Real Amplitudes (zob. [manuskrypt](https://arxiv.org/pdf/2412.19150)). W związku z tym, dla dużych problemów, Optimized Real Amplitudes wymaga więcej uruchomień Circuit. Optimized Real Amplitudes będzie prawdopodobnie przydatny dla problemów wymagających do 100 Qubitów, jednak zaleca się ostrożność przy ustawianiu parametrów ``population_size``. Jako przykład tego skalowania ``population_size``, poprzednia tabela pokazuje, że dla problemu z 84 Qubitami Optimized Real Amplitudes wymaga ``population_size`` równego 120, natomiast dla problemu z 56 Qubitami wystarczy ``population_size`` równy 40.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

Możliwe jest również wybranie konkretnego ansatzu. Poniżej użyto ansatzu ``'Tailored'``.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. Uruchom problem.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Pobierz wyniki
Funkcja zwraca słownik z trajektoriami inwestycji uporządkowanymi od najniższej do najwyższej wartości funkcji celu (zob. sekcję [Dane wyjściowe](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) w dokumentacji API). Ten zestaw wyników umożliwia identyfikację trajektorii o najniższym koszcie oraz odpowiadających jej ocen inwestycji. Dodatkowo pozwala na analizę różnych trajektorii, ułatwiając wybór tych, które najlepiej odpowiadają konkretnym potrzebom lub celom. Ta elastyczność sprawia, że wybory można dostosować do różnorodnych preferencji lub scenariuszy.
Zacznij od przedstawienia strategii wynikowej, która osiągnęła najniższy koszt celu znaleziony podczas procesu.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


Następnie, korzystając z metadanych, możesz uzyskać dostęp do wyników wszystkich próbkowanych strategii. Dzięki temu możesz przeprowadzić dalszą analizę alternatywnych trajektorii zwróconych przez optymalizator. W tym celu odczytaj słownik zapisany w `dpo_result['metadata']['all_samples_metrics']`, który zawiera nie tylko dodatkowe informacje o strategii optymalnej, ale także szczegóły dotyczące innych strategii kandydujących ocenianych podczas optymalizacji.

Poniższy przykład pokazuje, jak odczytać te informacje za pomocą `pandas`, aby wyodrębnić kluczowe metryki powiązane z optymalną strategią. Obejmują one Odchylenie od Ograniczeń (Restriction Deviation), Wskaźnik Sharpe'a (Sharpe Ratio) oraz odpowiadający zwrot z inwestycji.